# Cleaning and Filtering DistillGPT-2 Generated Synthetic SMS Spam Dataset


In [1]:
# Import required libraries
import pandas as pd
import re
import csv

input_path = "DistilGPT2_Synthetic_SMSSpamCollection.csv"

# Load the data
df = pd.read_csv(input_path, sep='\t', header=None, names=["label", "message"])
print(f"Loaded {len(df)} rows from {input_path}")


Loaded 4000 rows from DistilGPT2_Synthetic_SMSSpamCollection.csv


In [2]:
# Basic cleanup: drop empty or null messages
df.dropna(inplace=True)
df['message'] = df['message'].astype(str).str.strip()
df = df[df['message'].str.len() > 0]


In [3]:
# Extended spam keyword list based on common spam patterns
spam_keywords = [
    'win', 'won', 'winner', 'free', 'claim', 'click', 'urgent', 'now', 'congratulations',
    'offer', 'bonus', 'reward', 'deal', 'exclusive', 'gift', 'limited', 'act now',
    'hurry', 'last chance', 'subscribe', 'order now', 'apply now',

    'cash', 'money', 'loan', 'credit', 'bank', 'account', 'balance', 'payment', 'transfer',
    'income', 'rich', 'earn', 'investment', 'refund', 'save big', 'discount', 'price',

    'buy', 'purchase', 'promo', 'promotion', 'sale', 'shopping', 'store', 'shop now',
    'checkout', 'get yours', 'online store', 'special offer', 'limited time',

    'txt', 'text', 'sms', 'call', 'toll free', 'hotline', 'number', 'contact', 'unsubscribe',

    'www', 'http', '.com', '.net', '.org',

    'risk-free', 'guarantee', 'no cost', 'no obligation', 'as seen on', 'trial', 'exclusive deal'

]


In [4]:
# Helper regex to catch special character-only messages
special_char_pattern = re.compile(r'^[^a-zA-Z0-9]+$')

# Function to check general validity of a message
def is_valid_message(text):
    if text is None:
        return False
    text = text.strip()
    if text == '':
        return False
    if len(text) < 5:  # Too short
        return False
    if len(text.split()) < 2:  # Not enough words
        return False
    if special_char_pattern.match(text):  # Only special characters
        return False
    return True


In [5]:
# Spam validator: must be valid and contain spam keywords
def is_valid_spam(text):
    if not is_valid_message(text):
        return False
    if not any(keyword in text.lower() for keyword in spam_keywords):
        return False
    return True

# Ham validator: must be valid and NOT have too many spam terms
def is_valid_ham(text):
    if not is_valid_message(text):
        return False
    spammy_terms = sum(1 for keyword in spam_keywords if keyword in text.lower())
    if spammy_terms > 6:  # More than 6 spam terms? Probably not ham
        return False
    return True


In [6]:
# Filter the dataset using the above rules
filtered_data = []

for idx, row in df.iterrows():
    label = row['label'].strip().lower()
    message = row['message'].strip()

    if label == 'spam' and is_valid_spam(message):
        filtered_data.append((label, message))
    elif label == 'ham' and is_valid_ham(message):
        filtered_data.append((label, message))

print(f"Kept {len(filtered_data)} messages after filtering out invalid or misaligned examples.")


Kept 2440 messages after filtering out invalid or misaligned examples.


In [7]:
from collections import Counter

# Count how many 'spam' and 'ham' messages are in the filtered dataset
label_counts = Counter(label for label, _ in filtered_data)

print("📊 Message Counts:")
print(f"🟩 Ham:  {label_counts.get('ham', 0)}")
print(f"🟥 Spam: {label_counts.get('spam', 0)}")


📊 Message Counts:
🟩 Ham:  1734
🟥 Spam: 706


In [8]:
# Save cleaned data back to a new CSV (tab-separated) WITHOUT header
output_path = "Cleaned_DistillGPT2_Synthetic_SMSSpamCollection.csv"

with open(output_path, "w", encoding="utf-8", newline='') as f:
    writer = csv.writer(f, delimiter='\t')
    # No header row this time
    for label, message in filtered_data:
        writer.writerow([label, message])

print(f"Cleaned data saved to: {output_path} (no header)")


Cleaned data saved to: Cleaned_DistillGPT2_Synthetic_SMSSpamCollection.csv (no header)


In [9]:
# Merging data


# Load both cleaned datasets (assuming tab-separated with header)
distil_path = "Cleaned_DistillGPT2_Synthetic_SMSSpamCollection.csv"
gpt2_path = "Cleaned_GPT2_Synthetic_SMSSpamCollection.csv"

df_distil = pd.read_csv(distil_path, sep='\t')
df_gpt2 = pd.read_csv(gpt2_path, sep='\t')

print(f"DistilGPT-2 dataset size: {len(df_distil)}")
print(f"GPT-2 dataset size: {len(df_gpt2)}")

# Concatenate the two DataFrames
merged_df = pd.concat([df_distil, df_gpt2], ignore_index=True)

print(f"Total merged dataset size: {len(merged_df)}")

# Shuffle the merged data for randomness
merged_df = merged_df.sample(frac=1, random_state=42).reset_index(drop=True)

# Save merged dataset to new CSV (tab-separated, with header)
merged_output_path = "Merged_Cleaned_Synthetic_SMSSpamCollection.csv"
merged_df.to_csv(merged_output_path, sep='\t', index=False)

print(f"Merged dataset saved to: {merged_output_path}")


DistilGPT-2 dataset size: 2439
GPT-2 dataset size: 2162
Total merged dataset size: 4601
Merged dataset saved to: Merged_Cleaned_Synthetic_SMSSpamCollection.csv
